# LSTM Autoencoder Training — ECG Anomaly Detection
**Unsupervised | Baseline Comparison vs Transformer AE | Trained on Normal Beats Only**
Threshold = mean + 2×std of validation reconstruction errors.

### 1. Mount Google Drive

In [ ]:
# Local execution on Windows
print('Local execution mode')

### 2. Verify GPU

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0) if device.type == "cuda" else "NOT FOUND — Runtime > Change runtime type > T4 GPU"}')

### 3. Install & Import

In [ ]:
!pip install scikit-learn seaborn --quiet

import sys, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from sklearn.metrics import classification_report, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

sys.path.append(r'C:/ecg_arrhythmia/src/layer3_models')
from lstm_ae import LSTMAutoencoder

### 4. Paths & Config

In [ ]:
DATA_DIR  = Path(r'D:/ecg_project/datasets/mitbih/processed')
CKPT_BASE = Path(r'D:/ecg_project/models/checkpoints/lstm_ae')
BEST_DIR  = CKPT_BASE / 'best'
EPOCH_DIR = CKPT_BASE / 'epochs'
PLOT_DIR  = CKPT_BASE / 'plots'

for d in [BEST_DIR, EPOCH_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BATCH_SIZE   = 256
EPOCHS       = 100
LR           = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE     = 10
CKPT_EVERY   = 5
VAL_SPLIT    = 0.1
NORMAL_LABEL = 0

HIDDEN_SIZE  = 64
NUM_LAYERS   = 2
DROPOUT      = 0.2

SHORT_NAMES = ['N', 'L', 'R', 'A', 'V', '/', 'E', 'F']
CLASS_NAMES = {
    0: 'Normal (N)',
    1: 'Left Bundle Branch Block (L)',
    2: 'Right Bundle Branch Block (R)',
    3: 'Atrial Premature Beat (A)',
    4: 'Premature Ventricular Contraction (V)',
    5: 'Paced Beat (/)',
    6: 'Ventricular Escape Beat (E)',
    7: 'Fusion Beat (F)',
}
NUM_CLASSES = len(CLASS_NAMES)

### 5. Class Distribution (Normal vs Anomaly)

In [ ]:
train_labels = np.load(DATA_DIR / 'train' / 'labels.npy')
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

colors = ['steelblue' if i == NORMAL_LABEL else 'tomato' for i in range(NUM_CLASSES)]
plt.figure(figsize=(10, 4))
bars = plt.bar(SHORT_NAMES, class_counts, color=colors, edgecolor='black')
for bar, count in zip(bars, class_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'{int(count)}', ha='center', va='bottom', fontsize=9)
plt.title('Class Distribution — Blue=Normal (Training), Red=Anomaly (Eval Only)')
plt.xlabel('Beat Class')
plt.ylabel('Sample Count')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'class_distribution.png', dpi=150)
plt.show()

normal_count = class_counts[NORMAL_LABEL]
print(f'Normal beats used for training : {normal_count}')
print(f'Anomaly beats used for eval only: {class_counts.sum() - normal_count}')

### 6. Dataset — Normal Beats Only

In [ ]:
class NormalBeatDataset(Dataset):
    """Loads only Normal (label=0) beats for unsupervised autoencoder training."""
    def __init__(self, beats):
        self.beats = torch.tensor(beats, dtype=torch.float32)
    def __len__(self):          return len(self.beats)
    def __getitem__(self, idx): return self.beats[idx]


class AllBeatsDataset(Dataset):
    """All beats with labels — used only for threshold evaluation and testing."""
    def __init__(self, split):
        self.beats  = torch.tensor(np.load(DATA_DIR / split / 'beats.npy'),  dtype=torch.float32)
        self.labels = torch.tensor(np.load(DATA_DIR / split / 'labels.npy'), dtype=torch.long)
    def __len__(self):          return len(self.labels)
    def __getitem__(self, idx): return self.beats[idx], self.labels[idx]


train_beats  = np.load(DATA_DIR / 'train' / 'beats.npy')
train_labels = np.load(DATA_DIR / 'train' / 'labels.npy')
X_normal     = train_beats[train_labels == NORMAL_LABEL]

val_size   = int(len(X_normal) * VAL_SPLIT)
train_size = len(X_normal) - val_size
train_ds, val_ds = random_split(NormalBeatDataset(X_normal), [train_size, val_size])

train_loader = DataLoader(train_ds,               batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,                 batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(AllBeatsDataset('test'), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

print(f'Train normal beats : {train_size:,}')
print(f'Val   normal beats : {val_size:,}')
print(f'Test  total beats  : {len(test_loader.dataset):,}')

### 7. Model, Loss & Optimizer

In [ ]:
model = LSTMAutoencoder(
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT,
).to(device)

model.model_summary()

criterion = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = GradScaler('cuda', enabled=(device.type == 'cuda'))

### 8. Training Loop

In [ ]:
best_val_loss = float('inf')
patience_ctr  = 0
history       = {'train_loss': [], 'val_loss': [], 'lr': []}

for epoch in range(1, EPOCHS + 1):

    model.train()
    train_loss = 0.0
    for beats in train_loader:
        beats = beats.to(device)
        optimizer.zero_grad()
        with autocast('cuda', enabled=(device.type == 'cuda')):
            loss = criterion(model(beats), beats)
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for beats in val_loader:
            beats = beats.to(device)
            val_loss += criterion(model(beats), beats).item()
    val_loss /= len(val_loader)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(current_lr)

    print(f'Epoch {epoch:03d} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | LR: {current_lr:.6f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_ctr  = 0
        torch.save(model.state_dict(), BEST_DIR / 'lstm_ae_best.pth')
        print(f'  → Best model saved (val_loss: {best_val_loss:.6f})')
    else:
        patience_ctr += 1

    if epoch % CKPT_EVERY == 0:
        torch.save(model.state_dict(), EPOCH_DIR / f'lstm_ae_epoch_{epoch}.pth')
        print(f'  → Checkpoint saved (epoch {epoch})')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nTraining complete | Best Val Loss: {best_val_loss:.6f}')

### 9. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train Loss', color='mediumseagreen')
ax1.plot(history['val_loss'],   label='Val Loss',   color='darkorange')
ax1.set_title('Reconstruction Loss (MSE)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.legend()

ax2.plot(history['lr'], color='green', marker='o', markersize=3)
ax2.set_title('Learning Rate Schedule (Cosine Annealing)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.set_yscale('log')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'training_curves.png', dpi=150)
plt.show()

### 10. Compute Anomaly Threshold on Validation Set

In [ ]:
model.load_state_dict(torch.load(BEST_DIR / 'lstm_ae_best.pth'))
model.eval()

val_errors = []
with torch.no_grad():
    for beats in val_loader:
        errors = model.reconstruction_error(beats.to(device))
        val_errors.extend(errors.cpu().numpy())

val_errors = torch.tensor(val_errors)
threshold  = model.compute_threshold(val_errors)
model.save_threshold(BEST_DIR / 'lstm_ae_threshold.json')

print(f'Val Error Mean : {val_errors.mean():.6f}')
print(f'Val Error Std  : {val_errors.std():.6f}')
print(f'Threshold      : {threshold:.6f}  (mean + 2×std)')

### 11. Threshold Distribution Plot

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(val_errors.numpy(), bins=60, color='mediumseagreen', edgecolor='none', alpha=0.8, label='Val Normal Errors')
plt.axvline(val_errors.mean().item(), color='green',  linestyle='--', linewidth=1.5, label=f'Mean = {val_errors.mean():.4f}')
plt.axvline(threshold,                 color='tomato', linestyle='--', linewidth=2.0, label=f'Threshold = {threshold:.4f}')
plt.title('Reconstruction Error Distribution — Validation Normal Beats')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'threshold_distribution.png', dpi=150)
plt.show()

### 12. Evaluate on Test Set

In [ ]:
all_errors, all_labels_list, all_preds = [], [], []

model.eval()
with torch.no_grad():
    for beats, labels in test_loader:
        errors = model.reconstruction_error(beats.to(device))
        preds  = (errors > threshold).long()
        all_errors.extend(errors.cpu().numpy())
        all_labels_list.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())

all_errors = np.array(all_errors)
all_labels = np.array(all_labels_list)
all_preds  = np.array(all_preds)

true_binary = (all_labels != NORMAL_LABEL).astype(int)

print(classification_report(true_binary, all_preds, target_names=['Normal', 'Anomaly'], digits=4))

try:
    roc_auc = roc_auc_score(true_binary, all_errors)
    print(f'ROC-AUC Score: {roc_auc:.4f}')
except Exception:
    pass

### 13. Reconstruction Error per Class

In [ ]:
print(f'{"Class":<45} {"Mean Error":>12} {"Std":>10} {"Detected %":>12}')
print('-' * 83)
for label_id, name in CLASS_NAMES.items():
    mask = all_labels == label_id
    if mask.sum() == 0:
        continue
    errs    = all_errors[mask]
    det_pct = 100.0 * (errs > threshold).mean()
    print(f'{name:<45} {errs.mean():>12.6f} {errs.std():>10.6f} {det_pct:>11.1f}%')

### 14. Reconstruction Error Distribution (Normal vs Anomaly)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(all_errors[all_labels == 0], bins=60, alpha=0.7, label='Normal',  color='mediumseagreen', edgecolor='none')
axes[0].hist(all_errors[all_labels != 0], bins=60, alpha=0.7, label='Anomaly', color='tomato',         edgecolor='none')
axes[0].axvline(threshold, color='black', linestyle='--', linewidth=1.5, label=f'Threshold={threshold:.4f}')
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Count')
axes[0].set_title('LSTM AE — Error Distribution')
axes[0].legend()

mean_errors = [all_errors[all_labels == i].mean() if (all_labels == i).sum() > 0 else 0
               for i in range(NUM_CLASSES)]
axes[1].barh(SHORT_NAMES, mean_errors, color='mediumseagreen', edgecolor='black')
axes[1].axvline(threshold, color='tomato', linestyle='--', linewidth=1.5, label=f'Threshold={threshold:.4f}')
axes[1].set_xlabel('Mean Reconstruction Error')
axes[1].set_title('Mean Error per Class')
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / 'error_distribution.png', dpi=150)
plt.show()

### 15. Per-Class Error Boxplot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

box_data   = [all_errors[all_labels == i] for i in range(NUM_CLASSES)]
box_colors = ['mediumseagreen' if i == 0 else 'tomato' for i in range(NUM_CLASSES)]

bp = ax.boxplot(box_data, patch_artist=True, notch=False, showfliers=False)
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.axhline(threshold, color='black', linestyle='--', linewidth=1.5, label=f'Threshold={threshold:.4f}')
ax.set_xticks(range(1, NUM_CLASSES + 1))
ax.set_xticklabels(SHORT_NAMES)
ax.set_xlabel('Beat Class')
ax.set_ylabel('Reconstruction Error (MSE)')
ax.set_title('LSTM AE — Reconstruction Error Spread per Class')
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'error_boxplot.png', dpi=150)
plt.show()

### 16. ROC Curve — Anomaly Detection

In [ ]:
fpr, tpr, _ = roc_curve(true_binary, all_errors)
roc_auc     = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='mediumseagreen', linewidth=2, label=f'LSTM AE (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — LSTM AE (Normal vs Anomaly)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'roc_auc.png', dpi=150)
plt.show()

### 17. Beat Reconstruction Overlay

In [ ]:
test_beats = np.load(DATA_DIR / 'test' / 'beats.npy')
test_lbls  = np.load(DATA_DIR / 'test' / 'labels.npy')

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

model.eval()
for class_idx in range(NUM_CLASSES):
    idx    = np.where(test_lbls == class_idx)[0][0]
    beat   = test_beats[idx]
    x_in   = torch.tensor(beat, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        recon = model(x_in).squeeze(0).cpu().numpy()
        error = float(model.reconstruction_error(x_in).cpu())

    is_anomaly = error > threshold
    color      = 'tomato' if is_anomaly else 'mediumseagreen'
    axes[class_idx].plot(beat,  color='mediumseagreen', linewidth=1.0, label='Original',      alpha=0.9)
    axes[class_idx].plot(recon, color='darkorange',      linewidth=1.0, label='Reconstructed', alpha=0.9, linestyle='--')
    axes[class_idx].set_title(
        f'Class {SHORT_NAMES[class_idx]} | Error: {error:.4f}\n{"ANOMALY" if is_anomaly else "Normal"}',
        fontsize=9, color=color
    )
    axes[class_idx].axis('off')

axes[0].legend(fontsize=7, loc='upper right')
plt.suptitle('Beat Reconstruction Overlay (Green=Original, Orange=Reconstructed)', fontsize=12)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'beat_reconstruction.png', dpi=150)
plt.show()

### 18. Save Final Model & Summary

In [ ]:
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'config': {
            'hidden_size': HIDDEN_SIZE,
            'num_layers' : NUM_LAYERS,
            'dropout'    : DROPOUT,
        },
        'threshold' : float(threshold),
        'val_mean'  : float(val_errors.mean()),
        'val_std'   : float(val_errors.std()),
    },
    BEST_DIR / 'lstm_ae_final.pth'
)

print('All files saved to Drive:')
print(f'  best/   → lstm_ae_best.pth, lstm_ae_final.pth, lstm_ae_threshold.json')
print(f'  epochs/ → lstm_ae_epoch_5.pth ... lstm_ae_epoch_N.pth')
print(f'  plots/  → class_distribution.png, training_curves.png, threshold_distribution.png,')
print(f'            error_distribution.png, error_boxplot.png, roc_auc.png, beat_reconstruction.png')
print(f'\nThreshold: {threshold:.6f}')
print(f'Next step: Phase 5 — Inference Pipeline')